In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# ==========================================
# HOUSE PRICE PREDICTION USING AMES DATASET
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==========================
# LOAD DATASET
# ==========================

df = pd.read_csv("AmesHousing.csv")

print("First 5 Rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

# ==========================
# DATA CLEANING
# ==========================

# Fill numeric missing values
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing Values After Cleaning:")
print(df.isnull().sum().sum())

# ==========================
# OUTLIER REMOVAL
# ==========================

# Handle outliers only in target variable (SalePrice)

Q1 = df["SalePrice"].quantile(0.25)
Q3 = df["SalePrice"].quantile(0.75)

IQR = Q3 - Q1

df = df[
    (df["SalePrice"] >= (Q1 - 1.5 * IQR)) &
    (df["SalePrice"] <= (Q3 + 1.5 * IQR))
]

print("\nShape After Outlier Removal:")
print(df.shape)

# ==========================
# FEATURE ENGINEERING
# ==========================

df = pd.get_dummies(
    df,
    drop_first=True
)

# ==========================
# FEATURE SELECTION
# ==========================

target = "SalePrice"

X = df.drop(target, axis=1)
y = df[target]

# ==========================
# TRAIN TEST SPLIT
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================
# LINEAR REGRESSION
# ==========================

lr = LinearRegression()
lr.fit(X_train, y_train)

# ==========================
# RIDGE REGRESSION
# ==========================

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# ==========================
# RANDOM FOREST REGRESSION
# ==========================

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

# ==========================
# EVALUATION FUNCTION
# ==========================

def evaluate_model(model, X_test, y_test):

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)

    rmse = np.sqrt(
        mean_squared_error(y_test, pred)
    )

    r2 = r2_score(y_test, pred)

    return mae, rmse, r2

# ==========================
# MODEL COMPARISON
# ==========================

results = []

models = {
    "Linear Regression": lr,
    "Ridge Regression": ridge,
    "Random Forest": rf
}

for name, model in models.items():

    mae, rmse, r2 = evaluate_model(
        model,
        X_test,
        y_test
    )

    results.append([
        name,
        mae,
        rmse,
        r2
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2 Score"
    ]
)

print("\nModel Comparison:")
print(results_df)

# ==========================
# R² SCORE VISUALIZATION
# ==========================

plt.figure(figsize=(8, 5))

plt.bar(
    results_df["Model"],
    results_df["R2 Score"]
)

plt.title("Model Comparison (R² Score)")
plt.ylabel("R² Score")
plt.show()

# ==========================
# FEATURE IMPORTANCE
# ==========================

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

top10 = importance.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top10["Feature"],
    top10["Importance"]
)

plt.title("Top 10 Important Features")
plt.xlabel("Importance")
plt.gca().invert_yaxis()

plt.show()

# ==========================
# CUSTOM PREDICTION
# ==========================

sample = X.iloc[[0]]

prediction = rf.predict(sample)

print("\nPredicted House Price:")
print(f"${prediction[0]:,.2f}")